# Implementasi Model ke Lingkungan Produksi

Catatan ini menguraikan tahapan akhir dari siklus hidup *machine learning*, yaitu memindahkan model dari lingkungan pengembangan (*development*) menuju lingkungan produksi (*production*). Model komputasional yang hanya tersimpan di dalam *notebook* tidak akan memberikan nilai bisnis nyata hingga model tersebut diintegrasikan dengan sistem, aplikasi, atau proses operasional pengguna.

Fokus utama pada modul ini mencakup teknik serialisasi model, perancangan simulasi *endpoint* prediksi, evaluasi *latency* dan *throughput*, pemantauan model secara berkala (*monitoring*), serta penerapan konsep integrasi dan peluncuran berkelanjutan (CI/CD/CT) dalam ekosistem *machine learning*.

#### **Tujuan Pembelajaran**
* Memahami urgensi dan komponen utama dalam arsitektur *deployment* model.
* Melakukan serialisasi dan persistensi objek model menggunakan pustaka `joblib` dan `pickle`.
* Mengukur dan membandingkan performa inferensi (*latency* dan *throughput*) antara prediksi tunggal dan prediksi kelompok (*batch*).
* Mengimplementasikan *Pipeline* produksi untuk mencegah *training-serving skew* (ketidaksesuaian prapemrosesan).
* Memantau degradasi performa model (*data drift*) dan memicu pelatihan ulang secara inkremental.
* Mengelola siklus hidup model melalui pencatatan metadata, versi pustaka, dan gerbang peluncuran (*deployment gate*).

In [58]:
# Menonaktifkan peringatan komputasi
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore")

# Memuat pustaka sistem dan utilitas
import os
import json
import time
import pickle
import platform
from pathlib import Path

# Memuat pustaka komputasi numerik dan visualisasi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Memuat pustaka serialisasi dan scikit-learn
import sklearn
from joblib import dump, load
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Konfigurasi dasar
np.random.seed(2024)
pd.set_option("display.max_columns", 120)

# Menyiapkan direktori untuk menyimpan artefak produksi
ARTIFACT_DIR = Path("model_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Pustaka dan Direktori Artefak berhasil diinisialisasi.")

Pustaka dan Direktori Artefak berhasil diinisialisasi.


## 1. Serialisasi Model (Model Serialization)

Serialisasi adalah proses mengubah struktur objek model di dalam RAM menjadi aliran bita (*byte stream*) agar dapat disimpan secara permanen ke dalam *disk*. Di ekosistem Python, terdapat dua metode utama:
* **`pickle`**: Pustaka bawaan Python yang sangat fleksibel.
* **`joblib`**: Pustaka yang sangat direkomendasikan untuk `scikit-learn` karena memiliki efisiensi kompresi yang jauh lebih tinggi ketika menangani objek yang memuat susunan matriks `NumPy` berskala besar.

Keberhasilan proses persistensi diukur dari konsistensi absolut: model yang dimuat ulang (*deserialized*) wajib menghasilkan output prediksi yang identik persis dengan model aslinya.

In [59]:
# 1. Pembuatan Dataset Baseline
X, y = make_classification(n_samples=1000, n_features=20, n_informative=12, random_state=2024)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=2024, stratify=y)

# 2. Pelatihan Model Regresi Logistik
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# 3. Proses Serialisasi (Penyimpanan ke Disk)
model_path = ARTIFACT_DIR / "logistic_model.joblib"
dump(clf, model_path)
print(f"Model berhasil disimpan pada: {model_path}")

# 4. Deserialisasi dan Validasi Konsistensi
clf_loaded = load(model_path)
sampel_uji = X_test[:5]

prediksi_asli = clf.predict(sampel_uji)
prediksi_muat_ulang = clf_loaded.predict(sampel_uji)

# Verifikasi matematis
assert np.array_equal(prediksi_asli, prediksi_muat_ulang), "Gagal: Prediksi tidak konsisten!"
print("Validasi Sukses: Prediksi model original dan model muat ulang terbukti konsisten.")

Model berhasil disimpan pada: model_artifacts/logistic_model.joblib
Validasi Sukses: Prediksi model original dan model muat ulang terbukti konsisten.


## 2. Metrik Kinerja Inferensi Produksi

Dalam sistem produksi, akurasi bukanlah satu-satunya parameter keberhasilan. Kapasitas sistem dalam melayani permintaan (*requests*) sangat krusial:
* **Latency:** Waktu yang dibutuhkan untuk memproses dan melayani **satu** permintaan prediksi.
* **Throughput:** Jumlah prediksi maksimum yang dapat diselesaikan oleh sistem dalam **satu detik**.

$$\text{Throughput} = \frac{\text{Jumlah Prediksi}}{\text{Total Waktu}}$$

Secara arsitektur, mengirimkan prediksi dalam bentuk kelompok (*Batch Prediction*) akan menghasilkan *throughput* yang jauh lebih tinggi dibandingkan memprosesnya satu per satu (*Single Prediction Loop*) karena pemrosesan kelompok meminimalisasi *overhead* fungsi dan memanfaatkan komputasi vektorisasi CPU.

In [60]:
benchmark_data = X_test[:200]

# 1. Benchmark: Pemrosesan Tunggal (Looping)
start_single = time.perf_counter()
single_preds = [clf_loaded.predict(row.reshape(1, -1))[0] for row in benchmark_data]
single_time = time.perf_counter() - start_single

# 2. Benchmark: Pemrosesan Batch (Vektorisasi)
start_batch = time.perf_counter()
batch_preds = clf_loaded.predict(benchmark_data)
batch_time = time.perf_counter() - start_batch

# Komparasi Throughput
throughput_single = len(benchmark_data) / single_time
throughput_batch = len(benchmark_data) / batch_time

benchmark_df = pd.DataFrame({
    "Mode Operasi": ["Prediksi Tunggal (Loop)", "Prediksi Batch (Vektorisasi)"],
    "Total Prediksi": [len(benchmark_data), len(benchmark_data)],
    "Total Waktu (Detik)": [single_time, batch_time],
    "Throughput (Prediksi/Detik)": [throughput_single, throughput_batch]
})

print("=== Hasil Benchmark Inferensi ===")
display(benchmark_df)
print("\nAnalisis: Prediksi Batch memberikan throughput komputasi yang berkali-kali lipat lebih superior.")

=== Hasil Benchmark Inferensi ===


,Mode Operasi,Total Prediksi,Total Waktu (Detik),Throughput (Prediksi/Detik)
0,Prediksi Tunggal (Loop),200,0.026014,7688.272236
1,Prediksi Batch (Vektorisasi),200,0.000240,833645.950943



Analisis: Prediksi Batch memberikan throughput komputasi yang berkali-kali lipat lebih superior.


## 3. Integrasi Pipeline dan Tata Kelola Model (*Model Governance*)

Mengekspor model tanpa tahap prapemrosesan adalah sebuah kesalahan fatal yang memicu **Training-Serving Skew** (ketidakcocokan fitur input pengguna dengan ekspektasi model). Seluruh prapemrosesan mutlak harus dibungkus dalam objek `Pipeline`.

Selain itu, tata kelola produksi mewajibkan penyimpanan artefak pendukung:
1. **Metadata:** Mencatat versi pustaka Python dan *scikit-learn* (mencegah korupsi saat *deserialisasi* di server berbeda).
2. **Validation Snapshot:** Bukti performa akurasi saat model diekspor.
3. **Deployment Gate:** Aturan logika bisnis (threshold) untuk menolak model yang tidak memenuhi standar kualitas minimum.

In [61]:
# 1. Konstruksi Pipeline Terintegrasi
produksi_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=50, random_state=2024))
])

produksi_pipe.fit(X_train, y_train)
akurasi_validasi = accuracy_score(y_test, produksi_pipe.predict(X_test))

# 2. Ekspor Pipeline Utuh
versi_model = "v1.0"
pipe_path = ARTIFACT_DIR / f"pipeline_produksi_{versi_model}.joblib"
dump(produksi_pipe, pipe_path)

# 3. Pencatatan Metadata Sistem
metadata = {
    "model_name": "RandomForest_Pipeline",
    "version": versi_model,
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "validation_accuracy": float(akurasi_validasi)
}

metadata_path = ARTIFACT_DIR / f"metadata_{versi_model}.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

# 4. Simulasi CI/CD Deployment Gate
THRESHOLD_AKURASI = 0.80

def deployment_gate(skor_validasi, threshold):
    if skor_validasi >= threshold:
        return "DISETUJUI (AUTO-DEPLOY)"
    return "DITOLAK (HALT DEPLOYMENT)"

keputusan = deployment_gate(akurasi_validasi, THRESHOLD_AKURASI)

print("=== Log Artefak Produksi ===")
print(f"Artefak Pipeline   : {pipe_path.name}")
print(f"Artefak Metadata   : {metadata_path.name}")
print(f"Akurasi Validasi   : {akurasi_validasi:.4f} (Threshold: {THRESHOLD_AKURASI})")
print(f"Keputusan Produksi : {keputusan}")

=== Log Artefak Produksi ===
Artefak Pipeline   : pipeline_produksi_v1.0.joblib
Artefak Metadata   : metadata_v1.0.json
Akurasi Validasi   : 0.7880 (Threshold: 0.8)
Keputusan Produksi : DITOLAK (HALT DEPLOYMENT)


## 4. Pemantauan Berkelanjutan dan Pelatihan Inkremental

Setelah model diluncurkan (live), distribusi data dari dunia nyata dapat berubah sewaktu-waktu. Fenomena ini disebut **Data Drift** (pergeseran input) atau **Concept Drift** (pergeseran hubungan input-target). Performa model dipastikan akan terdegradasi seiring waktu.

Sistem MLOps yang matang membutuhkan mekanisme pemantauan akurasi *batch* yang berjalan. Jika akurasi menukik di bawah ambang batas kritis, sistem akan memicu *retraining* (pelatihan ulang). Untuk data yang mengalir secara kontinu (*streaming data*), algoritma seperti `SGDClassifier` dapat diperbarui secara matematis tanpa melatih ulang dari nol menggunakan metode `partial_fit()`.

In [62]:
# Simulasi Data Awal dan Data Streaming yang masuk secara periodik
X_init, y_init = make_classification(n_samples=500, random_state=2024)
X_stream, y_stream = make_classification(n_samples=400, random_state=2025)

# Memecah aliran data menjadi 4 gelombang (batch)
batch_stream_X = np.array_split(X_stream, 4)
batch_stream_y = np.array_split(y_stream, 4)

# Inisialisasi Model Inkremental
model_streaming = SGDClassifier(loss="log_loss", random_state=2024, warm_start=True)
model_streaming.partial_fit(X_init, y_init, classes=np.unique(y_init))

# Pemantauan Performa Lintas Batch
AMBANG_KRITIS = 0.55
riwayat_performa = []

print("=== Log Pemantauan Akurasi Produksi (Streaming) ===")
for id_batch, (X_batch, y_batch) in enumerate(zip(batch_stream_X, batch_stream_y), start=1):
    prediksi_batch = model_streaming.predict(X_batch)
    akurasi_batch = accuracy_score(y_batch, prediksi_batch)

    status_drift = "WASPADA (DRIFT TERDETEKSI)" if akurasi_batch < AMBANG_KRITIS else "STABIL"

    print(f"Batch {id_batch} | Akurasi: {akurasi_batch:.2f} | Status: {status_drift}")
    riwayat_performa.append(akurasi_batch)

    # Memicu pelatihan inkremental jika performa anjlok
    if akurasi_batch < AMBANG_KRITIS:
        print(" -> Memicu partial_fit() untuk adaptasi model dengan data terbaru...")
        model_streaming.partial_fit(X_batch, y_batch)

=== Log Pemantauan Akurasi Produksi (Streaming) ===
Batch 1 | Akurasi: 0.49 | Status: WASPADA (DRIFT TERDETEKSI)
 -> Memicu partial_fit() untuk adaptasi model dengan data terbaru...
Batch 2 | Akurasi: 0.78 | Status: STABIL
Batch 3 | Akurasi: 0.74 | Status: STABIL
Batch 4 | Akurasi: 0.77 | Status: STABIL


## Kesimpulan

*Deployment* sistem *machine learning* lebih condong pada rekayasa perangkat lunak (*Software Engineering*) dibandingkan komputasi analitik murni. Mengemas sebuah model adalah proses membangun sistem berkelanjutan.

**Praktik Terbaik (Best Practices):**
1. Ekspor seluruh prapemrosesan di dalam kerangka `Pipeline` untuk menjaga presisi matematis masukan produksi.
2. Gunakan `joblib` ketimbang `pickle` untuk kompresi struktur `scikit-learn` yang optimal.
3. Selalu simpan spesifikasi metadata lingkungan (versi Python dan *scikit-learn*) bersama artefak algoritma.
4. Terapkan pemrosesan berkelompok (*Batch Inference*) untuk memaksimalkan *throughput* jika arsitektur sistem memungkinkan.
5. Sediakan rancangan pemantauan (*monitoring*) dan rencana pengembalian sistem (*rollback plan*), karena asumsi fundamental dalam produksi adalah distribusi data akan selalu bergeser pada waktunya.